# 03 · Ingesta del clima aeronáutico (METAR)

**Fuente:** Iowa Environmental Mesonet (Iowa State University), archivo histórico de reportes METAR. Un METAR es el
parte meteorológico que emite cada aeropuerto (normalmente cada hora) en **texto libre codificado**, por ejemplo:

`SBGR 010000Z 12009KT 9999 BKN013 OVC020 19/17 Q1018`

Es el dato **no estructurado** del proyecto: se guarda tal cual en bronze y se convierte en columnas más adelante.

**Qué hace este notebook:** arma la lista de aeropuertos a partir de los vuelos que ya están en bronze (LATAM en Brasil
y todos los de EE. UU.), descarga el METAR de cada uno para todo el periodo y lo guarda con su manifiesto.

**Requisito:** haber ejecutado antes los notebooks 01 y 02.

**Salida:** `bronze/metar/station=XXXX/metar_XXXX_AAAA-MM_AAAA-MM.csv`, la ficha de cada aeropuerto en
`bronze/ref/aeropuertos/aeropuertos_iem.csv` y sus manifiestos en `bronze/_manifiestos/`.

In [1]:
import os
from pathlib import Path

import yaml

# Carpeta del repositorio (este notebook está en notebooks/)
RAIZ = Path.cwd().parent
with open(RAIZ / "config" / "pipeline.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
tablas = cfg["lake"]["tablas"]

# En la EC2 la variable LAKE_BUCKET trae el nombre del bucket; en una laptop se trabaja sobre data/lake
BUCKET = os.environ.get("LAKE_BUCKET", "").strip()
RAIZ_LOCAL = RAIZ / cfg["lake"]["raiz_local"]
BASE = f"s3a://{BUCKET}" if BUCKET else RAIZ_LOCAL.resolve().as_uri()


def ruta(tabla: str) -> str:
    """Ruta completa de una tabla declarada en config/pipeline.yaml."""
    return f"{BASE}/{tablas[tabla]}"


def meses(desde: str, hasta: str) -> list[tuple[int, int]]:
    """Pares (año, mes) entre dos fechas AAAA-MM, ambas incluidas."""
    anio, mes = map(int, desde.split("-"))
    anio_fin, mes_fin = map(int, hasta.split("-"))
    salida = []
    while (anio, mes) <= (anio_fin, mes_fin):
        salida.append((anio, mes))
        anio, mes = (anio + 1, 1) if mes == 12 else (anio, mes + 1)
    return salida


DESDE, HASTA = cfg["periodo"]["inicio"], cfg["periodo"]["fin"]
print("Destino:", f"s3://{BUCKET}" if BUCKET else "data/lake (local)")
print("Periodo:", DESDE, "a", HASTA)

Destino: s3://latam-bigdata-lake-728fd64c
Periodo: 2023-01 a 2025-12


In [2]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

fuente = cfg["fuentes"]["metar"]
empresas_latam = sorted(cfg["latam"]["pasajeros"])

# En la EC2 Spark toma la configuración del clúster; en una laptop corre en modo local
constructor = SparkSession.builder.appName("latam-bigdata-ingesta-metar")
if not BUCKET:
    constructor = constructor.master("local[*]").config("spark.sql.shuffle.partitions", "8")
spark = constructor.config("spark.sql.session.timeZone", "UTC").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/25 05:39:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/25 05:39:30 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


26/11/25 05:39:32 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


26/11/25 05:39:34 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to logs/spark-events/app-20261125053931-0000.inprogress. This is unsupported


## Funciones de apoyo

Descargar con reintentos, calcular la huella (hash SHA-256) de cada archivo, subirlo a bronze y dejar su manifiesto.

In [3]:
import hashlib
import json
import shutil
import tempfile
import time
from datetime import UTC, datetime

import boto3
import requests

s3 = boto3.client("s3") if BUCKET else None


def descargar(url: str, destino: Path, params: dict | None = None, reintentos: int = 5) -> bool:
    """Descarga por partes con reintentos de espera creciente. Devuelve False si el recurso no existe (404)."""
    cabeceras = {"User-Agent": cfg["fuentes"]["user_agent"]}
    for intento in range(1, reintentos + 1):
        try:
            with requests.get(url, params=params, headers=cabeceras, stream=True, timeout=300) as r:
                if r.status_code == 404:
                    return False
                r.raise_for_status()
                with open(destino, "wb") as f:
                    for bloque in r.iter_content(1 << 20):
                        f.write(bloque)
            return True
        except (requests.RequestException, OSError) as error:
            espera = min(5 * 2**intento, 300)
            print(f"  falló la descarga ({intento}/{reintentos}): {error}; reintento en {espera} s")
            time.sleep(espera)
    raise RuntimeError(f"No se pudo descargar {url}")


def sha256(ruta_archivo: Path) -> str:
    h = hashlib.sha256()
    with open(ruta_archivo, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


def contar_lineas(ruta_archivo: Path) -> int:
    with open(ruta_archivo, "rb") as f:
        return sum(bloque.count(b"\n") for bloque in iter(lambda: f.read(1 << 20), b""))


def existe(clave: str) -> bool:
    """True si la clave ya está en bronze."""
    if BUCKET:
        return s3.list_objects_v2(Bucket=BUCKET, Prefix=clave, MaxKeys=1).get("KeyCount", 0) > 0
    return (RAIZ_LOCAL / clave).exists()


def guardar(archivo: Path, clave: str) -> None:
    """Sube el archivo al bucket (o lo copia a data/lake en local)."""
    if BUCKET:
        s3.upload_file(str(archivo), BUCKET, clave)
    else:
        destino = RAIZ_LOCAL / clave
        destino.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(archivo, destino)


def escribir_manifiesto(fuente: str, clave: str, archivo: Path, url: str) -> dict:
    """Guarda de dónde salió el archivo, su tamaño, sus líneas, su hash y la fecha de descarga."""
    manifiesto = {
        "fuente": fuente, "archivo": clave, "url": url,
        "bytes": archivo.stat().st_size, "lineas": contar_lineas(archivo), "sha256": sha256(archivo),
        "descargado_utc": datetime.now(UTC).isoformat(timespec="seconds"),
    }
    temporal = archivo.with_name(archivo.name + ".json")
    temporal.write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding="utf-8")
    guardar(temporal, f"{tablas['manifiestos']}/{fuente}/{archivo.name}.json")
    temporal.unlink()
    return manifiesto


def leer_manifiestos(fuente: str) -> list[dict]:
    """Todos los manifiestos de una fuente."""
    prefijo = f"{tablas['manifiestos']}/{fuente}/"
    if BUCKET:
        salida = []
        for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefijo):
            for objeto in pagina.get("Contents", []):
                salida.append(json.loads(s3.get_object(Bucket=BUCKET, Key=objeto["Key"])["Body"].read()))
        return salida
    return [json.loads(p.read_text(encoding="utf-8")) for p in sorted((RAIZ_LOCAL / prefijo).glob("*.json"))]

## 1. Aeropuertos de Brasil

Se leen los VRA de bronze y se toman los aeropuertos de origen y destino de los vuelos de pasajeros de LATAM.
VRA usa el código **ICAO** (4 letras, por ejemplo `SBGR`), el mismo que identifica la estación METAR.

In [4]:
lineas = spark.read.text(f"{ruta('bronze_vra')}/year=*/month=*/*.csv")
campos = F.split(F.regexp_replace("value", '"', ""), ";")
vuelos_br = (
    lineas.where(~F.col("value").contains("Atualizado") & ~F.col("value").contains("ICAO Empresa"))
    .select(campos[0].alias("empresa"), campos[4].alias("origen"), campos[5].alias("destino"))
    .where(F.col("empresa").isin(empresas_latam))
)
aeropuertos_br = (
    vuelos_br.select(F.col("origen").alias("aeropuerto")).union(vuelos_br.select(F.col("destino").alias("aeropuerto")))
    .groupBy("aeropuerto").count().orderBy(F.desc("count")).toPandas()
)
print(f"Aeropuertos con vuelos LATAM: {len(aeropuertos_br)}")
aeropuertos_br.head(10)

Aeropuertos con vuelos LATAM: 128


,aeropuerto,count
0,SBGR,410050
1,SBSP,241313
2,SBBR,159710
3,SBRJ,79751
4,SBGL,55722
5,SBFZ,54811
6,SBPA,45721
7,SBCT,43910
8,SBCF,42457
9,SBSV,41198


## 2. Aeropuertos de EE. UU.

BTS usa el código **IATA** (3 letras, por ejemplo `LAX`). El archivo de IEM lo acepta directamente para los
aeropuertos de EE. UU. continental y Alaska; para Hawái se prueba también `P` + IATA (por ejemplo `PHNL`), y los
territorios (Puerto Rico, Islas Vírgenes, Guam) usan los códigos de `metar.codigos_manuales` en `config/pipeline.yaml`.

In [5]:
bts = spark.read.option("header", True).csv(f"{ruta('bronze_bts')}/year=*/month=*/*.csv")
aeropuertos_us = (
    bts.select(F.col("Origin").alias("aeropuerto")).union(bts.select(F.col("Dest").alias("aeropuerto")))
    .groupBy("aeropuerto").count().orderBy(F.desc("count")).toPandas()
)
print(f"Aeropuertos en BTS: {len(aeropuertos_us)}")
aeropuertos_us.head(10)

Aeropuertos en BTS: 362


,aeropuerto,count
0,ATL,1985528
1,DEN,1830572
2,DFW,1829318
3,ORD,1731810
4,CLT,1216159
5,LAX,1157795
6,PHX,1133980
7,LAS,1127862
8,SEA,984698
9,MCO,967920


## 3. Descarga del METAR de una estación

Se pide a IEM el periodo completo en hora UTC, solo los reportes rutinarios y especiales. Entre descargas hay una pausa
para no saturar el servicio, que es público y gratuito.

In [6]:
anio_ini, mes_ini = map(int, DESDE.split("-"))
anio_fin, mes_fin = map(int, HASTA.split("-"))
fin_exclusivo = (anio_fin + 1, 1) if mes_fin == 12 else (anio_fin, mes_fin + 1)


def parametros(estacion: str) -> dict:
    return {
        "station": estacion, "data": "metar", "tz": "Etc/UTC", "format": "onlycomma",
        "year1": anio_ini, "month1": mes_ini, "day1": 1,
        "year2": fin_exclusivo[0], "month2": fin_exclusivo[1], "day2": 1,
        "latlon": "no", "elev": "no", "missing": "empty", "trace": "empty", "direct": "no",
        "report_type": [3, 4],
    }


def ingerir_estacion(codigo: str, candidatos: list[str]) -> dict | None:
    clave = f"{tablas['bronze_metar']}/station={codigo}/metar_{codigo}_{DESDE}_{HASTA}.csv"
    if existe(clave):
        return {"estado": "ya estaba"}
    with tempfile.TemporaryDirectory() as carpeta:
        archivo = Path(carpeta) / f"metar_{codigo}_{DESDE}_{HASTA}.csv"
        for estacion in candidatos:
            descargar(fuente["url"], archivo, params=parametros(estacion), reintentos=fuente["reintentos"])
            time.sleep(fuente["pausa_seg"])
            with open(archivo, encoding="utf-8") as f:
                encabezado, primera = f.readline().strip(), f.readline().strip()
            if encabezado == "station,valid,metar" and primera:
                guardar(archivo, clave)
                url = f"{fuente['url']}?station={estacion}"
                return escribir_manifiesto("metar", clave, archivo, url)
    return None

## 4. Ejecutar todas las estaciones

Son varios cientos de estaciones: puede tardar entre 1 y 2 horas. Si se interrumpe, al volver a ejecutar esta celda
salta las que ya están en bronze.

In [7]:
estaciones = [(a, [a]) for a in aeropuertos_br["aeropuerto"]]
manuales = fuente["codigos_manuales"]
estaciones += [(a, [manuales[a]] if a in manuales else [a, "P" + a]) for a in aeropuertos_us["aeropuerto"]]

sin_datos = []
for i, (codigo, candidatos) in enumerate(estaciones, 1):
    resultado = ingerir_estacion(codigo, candidatos)
    if resultado is None:
        sin_datos.append(codigo)
    if i % 25 == 0:
        print(f"{i}/{len(estaciones)} estaciones revisadas; sin datos hasta ahora: {len(sin_datos)}")
print(f"Listo. Sin METAR disponible: {len(sin_datos)} -> {sin_datos}")

25/490 estaciones revisadas; sin datos hasta ahora: 0


50/490 estaciones revisadas; sin datos hasta ahora: 2


  falló la descarga (1/5): ("Connection broken: ConnectionResetError(104, 'Connection reset by peer')", ConnectionResetError(104, 'Connection reset by peer')); reintento en 10 s


75/490 estaciones revisadas; sin datos hasta ahora: 4


100/490 estaciones revisadas; sin datos hasta ahora: 7


125/490 estaciones revisadas; sin datos hasta ahora: 8


... (19 outputs left out)

  falló la descarga (1/5): ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')); reintento en 10 s


  falló la descarga (1/5): ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer')); reintento en 10 s


475/490 estaciones revisadas; sin datos hasta ahora: 34


Listo. Sin METAR disponible: 37 -> ['SBVC', 'SBSI', 'SBJA', 'SDSC', 'SBZM', 'SBDO', 'SBDB', 'SACE', 'PBI', 'OGG', 'KOA', 'LIH', 'ITO', 'AZA', 'JNU', 'FCA', 'FAI', 'KTN', 'HHH', 'YUM', 'SIT', 'MQT', 'SCE', 'USA', 'PSG', 'WRG', 'YAK', 'CDV', 'BET', 'SCC', 'OME', 'OTZ', 'BRW', 'CLD', 'DLG', 'GUF', 'GST']


## 5. Ficha de cada aeropuerto

Para pasar las horas de BTS (hora local de cada aeropuerto) a UTC hace falta la zona horaria de cada aeropuerto.
IEM publica la ficha de cada estación: nombre, país, coordenadas y zona horaria IANA (por ejemplo
`America/Los_Angeles`). Se busca la ficha de **todos** los aeropuertos de los vuelos, tengan o no METAR: sin zona
horaria un vuelo de EE. UU. no se puede pasar a UTC. Las pocas zonas horarias que IEM trae mal se corrigen con
`metar.zonas_horarias_manuales` de `config/pipeline.yaml`.

In [8]:
import json
from urllib.parse import parse_qs, urlparse

clave_fichas = f"{tablas['bronze_aeropuertos']}/aeropuertos_iem.csv"
manifiestos = pd.DataFrame(leer_manifiestos("metar"))
manifiestos["aeropuerto"] = manifiestos["archivo"].str.extract(r"station=([^/]+)/")
manifiestos["estacion"] = manifiestos["url"].map(lambda u: parse_qs(urlparse(u).query)["station"][0])

# Estación con METAR ya resuelta por la ingesta; para el resto se prueban los candidatos
resueltas = dict(zip(manifiestos["aeropuerto"], manifiestos["estacion"]))


def buscar_ficha(estacion: str, carpeta: str) -> dict | None:
    archivo = Path(carpeta) / "estacion.json"
    encontrada = descargar(fuente["url_estacion"].format(estacion=estacion), archivo, reintentos=fuente["reintentos"])
    time.sleep(1)
    if not encontrada:
        return None
    datos = json.loads(archivo.read_text(encoding="utf-8")).get("data") or []
    return datos[0] if datos else None


fichas, sin_ficha = [], []
with tempfile.TemporaryDirectory() as carpeta:
    for aeropuerto, candidatos in estaciones:
        opciones = [resueltas[aeropuerto]] if aeropuerto in resueltas else candidatos + ["K" + aeropuerto]
        for estacion in opciones:
            d = buscar_ficha(estacion, carpeta)
            if d and d.get("tzname"):
                fichas.append({"aeropuerto": aeropuerto, "estacion": estacion, "nombre": d["name"],
                               "pais": d["country"], "latitud": d["latitude"], "longitud": d["longitude"],
                               "zona_horaria": d["tzname"], "tiene_metar": aeropuerto in resueltas})
                break
        else:
            sin_ficha.append(aeropuerto)
    fichas = pd.DataFrame(fichas)
    for aeropuerto, zona in fuente["zonas_horarias_manuales"].items():
        fichas.loc[fichas["aeropuerto"] == aeropuerto, "zona_horaria"] = zona
    tabla = Path(carpeta) / "aeropuertos_iem.csv"
    fichas.to_csv(tabla, index=False)
    guardar(tabla, clave_fichas)
    escribir_manifiesto("aeropuertos", clave_fichas, tabla, fuente["url_estacion"])

print(f"Fichas: {len(fichas)} (con METAR: {fichas['tiene_metar'].sum()}) · sin ficha: {len(sin_ficha)} -> {sin_ficha}")
fichas.head(10)

Fichas: 468 (con METAR: 449) · sin ficha: 22 -> ['SBVC', 'SBSI', 'SBJA', 'SDSC', 'SBZM', 'SBDO', 'SBDB', 'KRSW', 'KBFM', 'KMZJ', 'KVCV', 'SACE', 'KOA', 'AZA', 'FCA', 'HHH', 'SCE', 'USA', 'CLD', 'DLG', 'GUF', 'GST']


,aeropuerto,estacion,nombre,pais,latitud,longitud,zona_horaria,tiene_metar
0,SBGR,SBGR,Sao Paulo,BR,-23.43208,-46.46951,America/Sao_Paulo,True
1,SBSP,SBSP,Sao Paulo,BR,-23.62669,-46.65537,America/Sao_Paulo,True
2,SBBR,SBBR,Brasilia,BR,-15.86667,-47.93333,America/Sao_Paulo,True
3,SBRJ,SBRJ,Rio De Janeiro,BR,-22.91046,-43.16313,America/Sao_Paulo,True
4,SBGL,SBGL,Rio De Janeiro,BR,-22.80890,-43.24365,America/Sao_Paulo,True
5,SBFZ,SBFZ,Fortaleza,BR,-3.77628,-38.53256,America/Fortaleza,True
6,SBPA,SBPA,Porto Alegre,BR,-29.99443,-51.17143,America/Sao_Paulo,True
7,SBCT,SBCT,Curitiba,BR,-25.52848,-49.17578,America/Sao_Paulo,True
8,SBCF,SBCF,Belo Horizonte,BR,-19.93447,-43.93561,America/Sao_Paulo,True
9,SBSV,SBSV,Salvador,BR,-12.91099,-38.33104,America/Bahia,True


## 6. Verificación

In [9]:
resumen = pd.DataFrame(leer_manifiestos("metar")).sort_values("archivo")
resumen["MB"] = (resumen["bytes"] / 1e6).round(2)
print(f"Estaciones: {len(resumen)} · Reportes: {resumen['lineas'].sum():,} · Tamaño: {resumen['bytes'].sum() / 1e9:.2f} GB")
resumen[["archivo", "lineas", "MB", "descargado_utc"]]

Estaciones: 453 · Reportes: 14,746,781 · Tamaño: 1.44 GB


,archivo,lineas,MB,descargado_utc
0,bronze/metar/station=ABE/metar_ABE_2023-01_202...,32193,3.27,2026-11-25T06:20:14+00:00
1,bronze/metar/station=ABI/metar_ABI_2023-01_202...,30870,3.13,2026-11-25T06:54:10+00:00
2,bronze/metar/station=ABQ/metar_ABQ_2023-01_202...,27898,3.13,2026-11-25T06:06:54+00:00
3,bronze/metar/station=ABR/metar_ABR_2023-01_202...,33422,3.56,2026-11-25T07:06:35+00:00
4,bronze/metar/station=ABY/metar_ABY_2023-01_202...,32667,3.32,2026-11-25T07:05:17+00:00
...,...,...,...,...
448,bronze/metar/station=VRB/metar_VRB_2023-01_202...,32244,3.31,2026-11-25T07:30:14+00:00
449,bronze/metar/station=WIII/metar_WIII_2023-01_2...,52261,4.08,2026-11-25T05:58:23+00:00
450,bronze/metar/station=WYS/metar_WYS_2023-01_202...,40055,3.25,2026-11-25T07:22:22+00:00
451,bronze/metar/station=XNA/metar_XNA_2023-01_202...,31642,3.19,2026-11-25T06:11:04+00:00


In [10]:
spark.stop()